# AdaFuse — Model Implementations

This notebook implements all models used in the AdaFuse BTP project:

| # | Model | Type | Role |
|---|-------|------|------|
| 1 | **MF-BPR** | Matrix Factorization | Baseline |
| 2 | **LightGCN** | GNN (He et al., SIGIR 2020) | Baseline / AdaFuse component |
| 3 | **SASRec** | Transformer (Kang & McAuley, IEEE ICDM 2018) | Baseline / AdaFuse component |
| 4 | **SASGCN** | GCN + SASRec, static fusion (Seng et al., IEEE Access 2024) | Direct comparator |
| 5 | **AdaFuse** | LightGCN + SASRec, adaptive attention fusion (**Ours**) | Proposed model |

**Shared settings:**
- Embedding dimension: 64
- Training loss: BPR (Bayesian Personalized Ranking)
- Evaluation: Recall@K, NDCG@K at K ∈ {10, 20}
- Seeds: [42, 123, 456, 789, 1024]

---

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.sparse as sp
from typing import Optional, Tuple

# Try PyG, fall back to pure PyTorch sparse ops
try:
    import torch_geometric
    HAS_PYG = True
    print(f"PyTorch Geometric {torch_geometric.__version__} available")
except ImportError:
    HAS_PYG = False
    print("PyTorch Geometric not found — using pure PyTorch sparse ops")

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBED_DIM = 64
SEEDS = [42, 123, 456, 789, 1024]

---
## 2. BPR Loss & Utilities

**Bayesian Personalized Ranking (Rendle et al., UAI 2009)**

For a triple $(u, i^+, i^-)$ where $i^+$ is a positive (observed) item and $i^-$ is a sampled negative:

$$\mathcal{L}_{\text{BPR}} = -\frac{1}{|\mathcal{D}|} \sum_{(u, i^+, i^-) \in \mathcal{D}} \ln \sigma\big(\hat{y}_{u,i^+} - \hat{y}_{u,i^-}\big)$$

With L2 regularization:

$$\mathcal{L} = \mathcal{L}_{\text{BPR}} + \lambda \|\Theta\|^2$$

In [ ]:
def bpr_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> torch.Tensor:
    """BPR pairwise ranking loss."""
    return -F.logsigmoid(pos_scores - neg_scores).mean()


def l2_reg(*embeddings: torch.Tensor) -> torch.Tensor:
    """L2 regularization over a set of embedding tensors."""
    reg = torch.tensor(0.0, device=embeddings[0].device)
    for emb in embeddings:
        reg = reg + emb.norm(2).pow(2)
    return reg / embeddings[0].shape[0]


def set_seed(seed: int):
    """Set all random seeds for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

### Graph Utility: Normalized Adjacency Matrix

For LightGCN and SASGCN we need the symmetrically normalized adjacency:

$$\tilde{A} = D^{-1/2} A D^{-1/2}$$

where $A$ is the $(|U|+|I|) \times (|U|+|I|)$ adjacency of the bipartite user-item graph with bidirectional edges, and $D$ is its degree matrix.

In [ ]:
def build_normalized_adj(num_users: int, num_items: int,
                         train_interactions: list[tuple[int, int]],
                         device: torch.device = DEVICE) -> torch.Tensor:
    """
    Build symmetric normalized adjacency for the user-item bipartite graph.

    Item node IDs are offset by num_users so that user i and item j occupy
    distinct rows/cols in the (num_users+num_items) square matrix.

    Returns a sparse torch tensor on `device`.
    """
    n = num_users + num_items

    rows, cols = [], []
    for u, i in train_interactions:
        item_node = i + num_users  # offset
        rows += [u, item_node]
        cols += [item_node, u]

    vals = np.ones(len(rows), dtype=np.float32)
    adj = sp.coo_matrix((vals, (rows, cols)), shape=(n, n))

    # D^{-1/2}
    degrees = np.array(adj.sum(axis=1)).flatten()
    d_inv_sqrt = np.where(degrees > 0, np.power(degrees, -0.5), 0.0)
    D_inv_sqrt = sp.diags(d_inv_sqrt)

    # D^{-1/2} A D^{-1/2}
    norm_adj = D_inv_sqrt @ adj @ D_inv_sqrt
    norm_adj = norm_adj.tocoo()

    indices = torch.LongTensor(np.stack([norm_adj.row, norm_adj.col]))
    values = torch.FloatTensor(norm_adj.data)
    sparse_tensor = torch.sparse_coo_tensor(indices, values, (n, n)).to(device)

    return sparse_tensor

---
## 3. Model 1: MF-BPR (Matrix Factorization)

The simplest baseline. Each user and item gets a learnable embedding vector. The predicted preference score is the dot product:

$$\hat{y}_{u,i} = \mathbf{e}_u^\top \mathbf{e}_i$$

Trained with BPR loss. No graph structure, no sequential information.

In [ ]:
class MFBPR(nn.Module):
    def __init__(self, num_users: int, num_items: int, embed_dim: int = EMBED_DIM):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embed_dim)
        self.item_embedding = nn.Embedding(num_items, embed_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

    def forward(self, users: torch.Tensor, pos_items: torch.Tensor,
                neg_items: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        user_emb = self.user_embedding(users)        # (B, d)
        pos_emb = self.item_embedding(pos_items)      # (B, d)
        neg_emb = self.item_embedding(neg_items)      # (B, d)

        pos_scores = (user_emb * pos_emb).sum(dim=1)  # (B,)
        neg_scores = (user_emb * neg_emb).sum(dim=1)  # (B,)

        return pos_scores, neg_scores

    def get_all_item_scores(self, users: torch.Tensor) -> torch.Tensor:
        """Scores for all items (used during evaluation)."""
        user_emb = self.user_embedding(users)          # (B, d)
        all_items = self.item_embedding.weight          # (|I|, d)
        return user_emb @ all_items.T                   # (B, |I|)

    def get_reg_loss(self, users, pos_items, neg_items, weight=1e-5):
        return weight * l2_reg(
            self.user_embedding(users),
            self.item_embedding(pos_items),
            self.item_embedding(neg_items),
        )

---
## 4. Model 2: LightGCN

**Reference:** He et al., *LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation*, SIGIR 2020.

LightGCN's key insight: for collaborative filtering, feature transformations and nonlinear activations in GCN **hurt** performance. The only useful operation is neighborhood aggregation.

### Propagation

$$\mathbf{E}^{(k)} = \tilde{A} \, \mathbf{E}^{(k-1)}$$

where $\tilde{A} = D^{-1/2} A D^{-1/2}$ is the normalized adjacency.

### Layer Combination

$$\mathbf{E}_{\text{final}} = \frac{1}{K+1} \sum_{k=0}^{K} \mathbf{E}^{(k)}$$

Uniform averaging of all layers (including the initial embeddings). The paper uses $K=3$ layers.

### What's removed vs. standard GCN
- No weight matrices $W^{(k)}$ (IDs have no concrete semantics to transform)
- No activation functions $\sigma$ (multiple nonlinear layers bring no benefit for CF)
- No self-connections (layer combination already captures self-information)

In [ ]:
class LightGCN(nn.Module):
    def __init__(self, num_users: int, num_items: int,
                 norm_adj: torch.Tensor,
                 embed_dim: int = EMBED_DIM,
                 num_layers: int = 3):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.num_layers = num_layers
        self.norm_adj = norm_adj  # precomputed sparse (num_users+num_items)^2

        self.embedding = nn.Embedding(num_users + num_items, embed_dim)
        nn.init.xavier_uniform_(self.embedding.weight)

    def compute_all_embeddings(self) -> torch.Tensor:
        """Run LightGCN propagation and return layer-averaged embeddings."""
        embs = [self.embedding.weight]  # E^(0)
        x = self.embedding.weight

        for _ in range(self.num_layers):
            x = torch.sparse.mm(self.norm_adj, x)  # E^(k) = A_norm @ E^(k-1)
            embs.append(x)

        # Uniform layer combination: mean of layers 0..K
        out = torch.stack(embs, dim=0).mean(dim=0)  # (N, d)
        return out

    def forward(self, users: torch.Tensor, pos_items: torch.Tensor,
                neg_items: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        all_emb = self.compute_all_embeddings()

        user_emb = all_emb[users]                                     # (B, d)
        pos_emb = all_emb[pos_items + self.num_users]                 # (B, d)
        neg_emb = all_emb[neg_items + self.num_users]                 # (B, d)

        pos_scores = (user_emb * pos_emb).sum(dim=1)
        neg_scores = (user_emb * neg_emb).sum(dim=1)

        return pos_scores, neg_scores

    def get_user_embedding(self, users: torch.Tensor) -> torch.Tensor:
        all_emb = self.compute_all_embeddings()
        return all_emb[users]

    def get_item_embedding(self, items: Optional[torch.Tensor] = None) -> torch.Tensor:
        all_emb = self.compute_all_embeddings()
        item_emb = all_emb[self.num_users:]  # all item embeddings
        if items is not None:
            return item_emb[items]
        return item_emb

    def get_all_item_scores(self, users: torch.Tensor) -> torch.Tensor:
        all_emb = self.compute_all_embeddings()
        user_emb = all_emb[users]                    # (B, d)
        item_emb = all_emb[self.num_users:]          # (|I|, d)
        return user_emb @ item_emb.T                  # (B, |I|)

    def get_reg_loss(self, users, pos_items, neg_items, weight=1e-5):
        # Regularize the initial (layer-0) embeddings, not the propagated ones
        u_emb0 = self.embedding(users)
        p_emb0 = self.embedding(pos_items + self.num_users)
        n_emb0 = self.embedding(neg_items + self.num_users)
        return weight * l2_reg(u_emb0, p_emb0, n_emb0)

---
## 5. Model 3: SASRec

**Reference:** Kang & McAuley, *Self-Attentive Sequential Recommendation*, IEEE ICDM 2018.

SASRec applies **causal (left-to-right) self-attention** over a user's interaction sequence. Each position attends only to itself and earlier positions.

### Architecture

1. **Input:** item embedding + learnable positional embedding
   $$\mathbf{H}^0 = [\mathbf{e}_{i_1} + \mathbf{p}_1, \;\ldots\;, \mathbf{e}_{i_t} + \mathbf{p}_t]$$

2. **Self-attention block** (repeated $L$ times):
   - Multi-head causal self-attention
   - Residual connection + LayerNorm
   - Point-wise FFN (two linear layers with ReLU)
   - Residual connection + LayerNorm

3. **Output:** The embedding at the last position $\mathbf{H}^L_t$ is the user's sequential representation.

### Why causal attention?
At each position $t$, the model should only see items $1, \ldots, t$ (not future items). This matches the sequential prediction task: given the past, predict the next item.

In [ ]:
class PointWiseFFN(nn.Module):
    """Two-layer FFN applied independently to each position."""
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.2):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.dropout(F.relu(self.fc1(x))))


class SASRecBlock(nn.Module):
    """One Transformer block: causal MHA + FFN, each with residual + LayerNorm."""
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.2):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.ffn = PointWiseFFN(d_model, d_model * 4, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor,
                key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Causal self-attention
        attn_out, _ = self.attn(
            x, x, x, attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=False
        )
        x = self.norm1(x + self.dropout1(attn_out))

        # FFN
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))

        return x


class SASRec(nn.Module):
    def __init__(self, num_items: int, embed_dim: int = EMBED_DIM,
                 max_len: int = 50, num_heads: int = 1,
                 num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.num_items = num_items
        self.max_len = max_len
        self.embed_dim = embed_dim

        # +1 for padding token (index 0)
        self.item_embedding = nn.Embedding(num_items + 1, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        self.emb_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            SASRecBlock(embed_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.item_embedding.weight[1:])  # skip padding
        nn.init.xavier_uniform_(self.pos_embedding.weight)

    def _causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """Upper-triangular mask filled with -inf (prevents attending to future)."""
        return torch.triu(
            torch.full((seq_len, seq_len), float('-inf'), device=device),
            diagonal=1
        )

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        """
        Args:
            seq: (B, L) item IDs (1-indexed; 0 = padding).

        Returns:
            (B, L, d) contextualized embeddings at each position.
        """
        B, L = seq.shape
        device = seq.device

        positions = torch.arange(L, device=device).unsqueeze(0).expand(B, -1)  # (B, L)

        x = self.item_embedding(seq) + self.pos_embedding(positions)
        x = self.emb_dropout(x)

        attn_mask = self._causal_mask(L, device)           # (L, L)
        padding_mask = (seq == 0)                           # (B, L)

        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=padding_mask)

        x = self.norm(x)  # (B, L, d)
        return x

    def get_seq_embedding(self, seq: torch.Tensor) -> torch.Tensor:
        """
        Get the user's sequential representation from the last valid position.

        Args:
            seq: (B, L) padded sequences.

        Returns:
            (B, d) last-position embedding for each user.
        """
        output = self.forward(seq)  # (B, L, d)

        # Find last non-padding position per user
        lengths = (seq != 0).sum(dim=1) - 1  # (B,), 0-indexed
        lengths = lengths.clamp(min=0)

        # Gather the output at the last valid position
        idx = lengths.unsqueeze(1).unsqueeze(2).expand(-1, 1, self.embed_dim)  # (B,1,d)
        last_emb = output.gather(1, idx).squeeze(1)  # (B, d)

        return last_emb

    def predict_next(self, seq: torch.Tensor, candidates: torch.Tensor) -> torch.Tensor:
        """
        Score candidate items given a sequence.

        Args:
            seq: (B, L) padded sequences.
            candidates: (B, C) candidate item IDs.

        Returns:
            (B, C) scores.
        """
        user_emb = self.get_seq_embedding(seq)          # (B, d)
        cand_emb = self.item_embedding(candidates)      # (B, C, d)
        scores = (user_emb.unsqueeze(1) * cand_emb).sum(dim=2)  # (B, C)
        return scores

    def get_all_item_scores(self, seq: torch.Tensor) -> torch.Tensor:
        """Score ALL items for evaluation (full-ranking)."""
        user_emb = self.get_seq_embedding(seq)          # (B, d)
        # skip padding token at index 0
        all_items = self.item_embedding.weight[1:]       # (|I|, d)
        return user_emb @ all_items.T                    # (B, |I|)

---
## 6. Model 4: SASGCN (Comparator)

**Reference:** Seng et al., *Self-Attention Based Sequential Recommendation With Graph Convolutional Networks*, IEEE Access 2024.

SASGCN is the paper we directly extend. It combines:
- **Vanilla GCN** (with feature transforms and ReLU) for collaborative filtering
- **SASRec** for sequential modeling
- **Static fusion**: concatenation of both representations → MLP → final embedding

### Key differences from AdaFuse:
1. Uses vanilla GCN, **not** LightGCN (GCN includes weight matrices and activations that LightGCN proved harmful)
2. Fusion is **static** (same MLP for all users) — no adaptive per-user weighting
3. No analysis of when each signal helps

In [ ]:
class VanillaGCN(nn.Module):
    """
    Standard GCN with feature transforms + ReLU (NOT LightGCN).
    This mirrors the GCN branch in SASGCN (IEEE Access 2024).
    """
    def __init__(self, num_users: int, num_items: int,
                 norm_adj: torch.Tensor,
                 embed_dim: int = EMBED_DIM,
                 num_layers: int = 2):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.norm_adj = norm_adj

        self.embedding = nn.Embedding(num_users + num_items, embed_dim)
        nn.init.xavier_uniform_(self.embedding.weight)

        self.layers = nn.ModuleList([
            nn.Linear(embed_dim, embed_dim) for _ in range(num_layers)
        ])

    def forward(self) -> torch.Tensor:
        x = self.embedding.weight
        for layer in self.layers:
            x = torch.sparse.mm(self.norm_adj, x)
            x = F.relu(layer(x))  # feature transform + activation
        return x


class SASGCN(nn.Module):
    """
    SASGCN: GCN + SASRec with static concatenation+MLP fusion.
    Reproduces the architecture from Seng et al. (IEEE Access 2024).
    """
    def __init__(self, num_users: int, num_items: int,
                 norm_adj: torch.Tensor,
                 embed_dim: int = EMBED_DIM,
                 gcn_layers: int = 2,
                 sasrec_layers: int = 2,
                 sasrec_heads: int = 1,
                 max_len: int = 50,
                 dropout: float = 0.2):
        super().__init__()
        self.num_users = num_users
        self.embed_dim = embed_dim

        # GCN branch (vanilla, with transforms + ReLU)
        self.gcn = VanillaGCN(num_users, num_items, norm_adj,
                              embed_dim, gcn_layers)

        # SASRec branch
        self.sasrec = SASRec(num_items, embed_dim, max_len,
                             sasrec_heads, sasrec_layers, dropout)

        # Static fusion: concat (2d) → MLP → d
        self.fusion_mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, user_ids: torch.Tensor, sequences: torch.Tensor,
                pos_items: torch.Tensor, neg_items: torch.Tensor
                ) -> Tuple[torch.Tensor, torch.Tensor]:
        # GCN branch: collaborative embeddings
        gcn_all = self.gcn()                              # (N, d)
        e_col = gcn_all[user_ids]                         # (B, d)

        # SASRec branch: sequential embeddings
        e_seq = self.sasrec.get_seq_embedding(sequences)  # (B, d)

        # Static fusion: concatenate + MLP
        fused = self.fusion_mlp(torch.cat([e_seq, e_col], dim=1))  # (B, d)

        # Score items
        pos_emb = gcn_all[pos_items + self.num_users]
        neg_emb = gcn_all[neg_items + self.num_users]

        pos_scores = (fused * pos_emb).sum(dim=1)
        neg_scores = (fused * neg_emb).sum(dim=1)

        return pos_scores, neg_scores

    def get_all_item_scores(self, user_ids: torch.Tensor,
                            sequences: torch.Tensor) -> torch.Tensor:
        gcn_all = self.gcn()
        e_col = gcn_all[user_ids]
        e_seq = self.sasrec.get_seq_embedding(sequences)
        fused = self.fusion_mlp(torch.cat([e_seq, e_col], dim=1))

        item_emb = gcn_all[self.num_users:]   # (|I|, d)
        return fused @ item_emb.T              # (B, |I|)

---
## 7. Model 5: AdaFuse (Our Proposed Model)

**AdaFuse: Adaptive Attention Fusion of Sequential and Collaborative Representations for Recommendation**

### Key improvements over SASGCN:
1. **LightGCN** instead of vanilla GCN (proven 16-22% better for CF)
2. **Learned per-user attention gate** instead of static MLP fusion
3. **Shared item embeddings** between branches for parameter efficiency

### Attention Fusion Gate

$$\mathbf{z}(u) = [\mathbf{e}_{\text{seq}} \;\|\; \mathbf{e}_{\text{col}} \;\|\; \mathbf{e}_{\text{seq}} \odot \mathbf{e}_{\text{col}}] \in \mathbb{R}^{3d}$$

$$\alpha(u) = \sigma(\mathbf{w}^\top \mathbf{z}(u) + b) \in (0, 1)$$

$$\mathbf{e}_{\text{final}}(u) = \alpha(u) \cdot \mathbf{e}_{\text{seq}} + (1 - \alpha(u)) \cdot \mathbf{e}_{\text{col}}$$

The element-wise product $\mathbf{e}_{\text{seq}} \odot \mathbf{e}_{\text{col}}$ captures the *interaction* between the two signals, not just their independent features.

### Why a scalar gate?
- **Interpretable:** $\alpha(u)$ directly tells us how much user $u$ relies on sequential vs. collaborative signal
- **Analyzable:** we can scatter-plot $\alpha$ vs. interaction count to test our hypothesis
- **Simple:** fewer parameters than a vector gate or multi-head gate (ablated separately)

In [ ]:
class AdaFuse(nn.Module):
    """
    AdaFuse: Adaptive Attention Fusion of SASRec + LightGCN.

    The two branches SHARE the item embedding table so they operate
    in the same embedding space.
    """
    def __init__(self, num_users: int, num_items: int,
                 norm_adj: torch.Tensor,
                 embed_dim: int = EMBED_DIM,
                 lgcn_layers: int = 3,
                 sasrec_layers: int = 2,
                 sasrec_heads: int = 1,
                 max_len: int = 50,
                 dropout: float = 0.2):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embed_dim = embed_dim

        # ── LightGCN branch ──
        self.lgcn_num_layers = lgcn_layers
        self.norm_adj = norm_adj

        # Shared user+item embedding for LightGCN (users 0..num_users-1,
        # items num_users..num_users+num_items-1)
        self.lgcn_embedding = nn.Embedding(num_users + num_items, embed_dim)
        nn.init.xavier_uniform_(self.lgcn_embedding.weight)

        # ── SASRec branch ──
        self.sasrec = SASRec(num_items, embed_dim, max_len,
                             sasrec_heads, sasrec_layers, dropout)

        # Share item embeddings: copy LightGCN's item portion into SASRec.
        # We tie them by making SASRec's item_embedding.weight[1:] point to
        # the same storage as lgcn_embedding.weight[num_users:].
        # NOTE: this is handled in _sync_item_embeddings() called before forward.

        # ── Attention fusion gate ──
        # Input: [e_seq; e_col; e_seq * e_col] → scalar alpha
        self.gate = nn.Linear(embed_dim * 3, 1)

    def _lgcn_propagate(self) -> torch.Tensor:
        """LightGCN multi-layer propagation with uniform averaging."""
        embs = [self.lgcn_embedding.weight]
        x = self.lgcn_embedding.weight
        for _ in range(self.lgcn_num_layers):
            x = torch.sparse.mm(self.norm_adj, x)
            embs.append(x)
        return torch.stack(embs, dim=0).mean(dim=0)

    def _sync_item_embeddings(self):
        """
        Copy LightGCN item embeddings into SASRec's item embedding table.
        This ensures both branches see the same item representations.

        We copy data rather than share parameters because the two embedding
        tables have different shapes (SASRec has a padding token at index 0).
        Gradients flow through both copies.
        """
        with torch.no_grad():
            lgcn_items = self.lgcn_embedding.weight[self.num_users:]  # (|I|, d)
            self.sasrec.item_embedding.weight[1:] = lgcn_items

    def forward(self, user_ids: torch.Tensor, sequences: torch.Tensor,
                pos_items: torch.Tensor, neg_items: torch.Tensor
                ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            user_ids:   (B,) user indices
            sequences:  (B, L) padded item sequences (1-indexed, 0=pad)
            pos_items:  (B,) positive item indices (0-indexed)
            neg_items:  (B,) negative item indices (0-indexed)

        Returns:
            pos_scores: (B,) BPR positive scores
            neg_scores: (B,) BPR negative scores
            alpha:      (B,) per-user fusion weight (for analysis)
        """
        # ── LightGCN branch ──
        lgcn_all = self._lgcn_propagate()                 # (N, d)
        e_col = lgcn_all[user_ids]                        # (B, d)

        # ── SASRec branch ──
        e_seq = self.sasrec.get_seq_embedding(sequences)  # (B, d)

        # ── Attention fusion gate ──
        gate_input = torch.cat([
            e_seq,
            e_col,
            e_seq * e_col  # element-wise interaction
        ], dim=1)  # (B, 3d)

        alpha = torch.sigmoid(self.gate(gate_input)).squeeze(-1)  # (B,)

        # Fuse: alpha * sequential + (1-alpha) * collaborative
        e_final = alpha.unsqueeze(1) * e_seq + (1 - alpha).unsqueeze(1) * e_col  # (B, d)

        # ── Score items ──
        pos_emb = lgcn_all[pos_items + self.num_users]    # (B, d)
        neg_emb = lgcn_all[neg_items + self.num_users]    # (B, d)

        pos_scores = (e_final * pos_emb).sum(dim=1)       # (B,)
        neg_scores = (e_final * neg_emb).sum(dim=1)       # (B,)

        return pos_scores, neg_scores, alpha

    def get_all_item_scores(self, user_ids: torch.Tensor,
                            sequences: torch.Tensor
                            ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Score all items for evaluation. Returns (scores, alpha)."""
        lgcn_all = self._lgcn_propagate()
        e_col = lgcn_all[user_ids]
        e_seq = self.sasrec.get_seq_embedding(sequences)

        gate_input = torch.cat([e_seq, e_col, e_seq * e_col], dim=1)
        alpha = torch.sigmoid(self.gate(gate_input)).squeeze(-1)

        e_final = alpha.unsqueeze(1) * e_seq + (1 - alpha).unsqueeze(1) * e_col

        item_emb = lgcn_all[self.num_users:]              # (|I|, d)
        scores = e_final @ item_emb.T                      # (B, |I|)

        return scores, alpha

    def get_reg_loss(self, user_ids, pos_items, neg_items, weight=1e-5):
        u_emb0 = self.lgcn_embedding(user_ids)
        p_emb0 = self.lgcn_embedding(pos_items + self.num_users)
        n_emb0 = self.lgcn_embedding(neg_items + self.num_users)
        return weight * l2_reg(u_emb0, p_emb0, n_emb0)

---
## 8. Model Summary

Compare parameter counts and architectural choices across all five models.

In [ ]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def print_model_summary(num_users: int = 52_024, num_items: int = 57_289):
    """
    Print a comparison table of all models.
    Uses Amazon-Beauty dimensions by default.
    """
    # Dummy adjacency for models that need it
    dummy_adj = torch.sparse_coo_tensor(
        indices=torch.LongTensor([[0], [0]]),
        values=torch.FloatTensor([0.0]),
        size=(num_users + num_items, num_users + num_items)
    )

    models = {
        "MF-BPR": MFBPR(num_users, num_items),
        "LightGCN": LightGCN(num_users, num_items, dummy_adj),
        "SASRec": SASRec(num_items),
        "SASGCN": SASGCN(num_users, num_items, dummy_adj),
        "AdaFuse (ours)": AdaFuse(num_users, num_items, dummy_adj),
    }

    print(f"{'Model':<20} {'Parameters':>12}  {'Type'}")
    print("-" * 60)
    for name, model in models.items():
        params = count_parameters(model)
        ptype = {
            "MF-BPR": "MF baseline",
            "LightGCN": "GNN baseline",
            "SASRec": "Transformer baseline",
            "SASGCN": "Hybrid comparator (IEEE 2024)",
            "AdaFuse (ours)": "Hybrid proposed",
        }[name]
        print(f"{name:<20} {params:>12,}  {ptype}")


print_model_summary()

### Architectural Comparison

| Feature | MF-BPR | LightGCN | SASRec | SASGCN | **AdaFuse** |
|---------|--------|----------|--------|--------|-------------|
| Sequential modeling | No | No | Yes (causal attention) | Yes (SASRec) | Yes (SASRec) |
| Collaborative signal | Dot product | Multi-hop GNN | No | GCN (with transforms) | **LightGCN** |
| Feature transforms | No | **No** (key insight) | Yes (FFN) | Yes (GCN layers) | FFN in SASRec only |
| Fusion | N/A | N/A | N/A | Concat + MLP (static) | **Attention gate (adaptive)** |
| Per-user adaptation | No | No | Implicit (via sequence) | No | **Yes ($\alpha$ per user)** |
| Shared item embeddings | N/A | N/A | N/A | No | **Yes** |

---

## Next Steps

These model classes are ready to be used with:
- `01_data_preprocessing.ipynb` — for data loading, graph construction, sequence building
- `03_training.ipynb` — for training loops, hyperparameter tuning, early stopping
- `04_evaluation.ipynb` — for Recall@K, NDCG@K, ablation studies, and alpha analysis